# VAR-d20 VAE Scale Fusion Ablation

This notebook tests which VAR VAE scales preserve content structure and which scales carry style-like color/texture information.

We do not modify the VAR transformer here. We only mix VAE token scales from a content image and a style image, then decode the mixed token pyramid.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/LeeHoang2710/Style-Transfer-Experiment.git"  # GitHub sync source
BRANCH = "main"
WORKSPACE = Path('/content/VAR_Style_Transfer_Workspace')

if not WORKSPACE.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(WORKSPACE)], check=True)
else:
    subprocess.run(['git', '-C', str(WORKSPACE), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(WORKSPACE), 'reset', '--hard', f'origin/{BRANCH}'], check=True)

print('Workspace synced to:')
subprocess.run(['git', '-C', str(WORKSPACE), 'log', '--oneline', '-1'], check=True)

os.chdir(WORKSPACE / 'VAR')
print(Path.cwd())


In [ ]:
!nvidia-smi
!pip install -q huggingface_hub einops matplotlib


In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download

weights_dir = Path('/content/VAR_weights')
weights_dir.mkdir(parents=True, exist_ok=True)

vae_path = hf_hub_download(
    repo_id='FoundationVision/var',
    filename='vae_ch160v4096z32.pth',
    local_dir=weights_dir,
)
var_path = hf_hub_download(
    repo_id='FoundationVision/var',
    filename='var_d20.pth',
    local_dir=weights_dir,
)


In [ ]:
import torch
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import torchvision.transforms.functional as TF
from torchvision.utils import make_grid
from models import build_vae_var

MODEL_DEPTH = 20
patch_nums = (1, 2, 3, 4, 5, 6, 8, 10, 13, 16)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

vae, var = build_vae_var(
    V=4096,
    Cvae=32,
    ch=160,
    share_quant_resi=4,
    device=device,
    patch_nums=patch_nums,
    num_classes=1000,
    depth=MODEL_DEPTH,
    shared_aln=False,
)

vae.load_state_dict(torch.load(vae_path, map_location='cpu'), strict=True)
var.load_state_dict(torch.load(var_path, map_location='cpu'), strict=True)
vae.eval(); var.eval()
for p in vae.parameters(): p.requires_grad_(False)
for p in var.parameters(): p.requires_grad_(False)

print(f'Loaded VAR-d{MODEL_DEPTH} on {device}')


## Load One Content/Style Pair

Start with one pair. Later, change `content_path` and `style_path` to repeat the same ablation on different objects and style types.

In [ ]:
DATA_ROOT = WORKSPACE
CONTENT_DIR = DATA_ROOT / 'content'
STYLE_DIR = DATA_ROOT / 'style'
OUT_DIR = Path('/content/VAR_outputs/vae_scale_fusion')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('content images:', len(sorted(CONTENT_DIR.glob('*.png'))))
print('style images:', len(sorted(STYLE_DIR.rglob('*.png'))))


In [ ]:
def load_var_image(path, size=256):
    img = Image.open(path).convert('RGB')
    img = ImageOps.fit(
        img,
        (size, size),
        method=Image.Resampling.LANCZOS,
        centering=(0.5, 0.5),
    )
    x = TF.to_tensor(img).mul(2).sub(1).unsqueeze(0).to(device)
    return x, img

def tensor_to_pil(x):
    x = x.detach().float().cpu()
    if x.ndim == 4:
        x = x[0]
    x = x.clamp(-1, 1).add(1).div(2)
    x = x.permute(1, 2, 0).numpy()
    return Image.fromarray((x * 255).astype('uint8'))

def show_tensor_img(x, title=None):
    plt.imshow(tensor_to_pil(x))
    if title:
        plt.title(title)
    plt.axis('off')

def show_pil_img(img, title=None):
    plt.imshow(img)
    if title:
        plt.title(title)
    plt.axis('off')


In [ ]:
content_path = CONTENT_DIR / '0001.png'
style_path = STYLE_DIR / 'VanGogh' / 'VanGogh001.png'

content_x, content_pil = load_var_image(content_path)
style_x, style_pil = load_var_image(style_path)

plt.figure(figsize=(6, 3))
plt.subplot(1, 2, 1); show_pil_img(content_pil, 'Content original')
plt.subplot(1, 2, 2); show_pil_img(style_pil, 'Style original')
plt.tight_layout()
plt.show()


## Tokenize And Reconstruct Baselines

This confirms both images can be represented by VAR's VAE token pyramid before we mix scales.

In [ ]:
with torch.no_grad():
    content_idx_Bl = vae.img_to_idxBl(content_x)
    style_idx_Bl = vae.img_to_idxBl(style_x)
    content_rec = vae.idxBl_to_img(content_idx_Bl, same_shape=True, last_one=True)
    style_rec = vae.idxBl_to_img(style_idx_Bl, same_shape=True, last_one=True)

for scale_id, (c_idx, s_idx, pn) in enumerate(zip(content_idx_Bl, style_idx_Bl, patch_nums)):
    print(
        f'scale {scale_id:02d}, patch={pn:02d}x{pn:02d}, '
        f'content={tuple(c_idx.shape)}, style={tuple(s_idx.shape)}'
    )


In [ ]:
plt.figure(figsize=(10, 5))
plt.subplot(2, 2, 1); show_pil_img(content_pil, 'Content original')
plt.subplot(2, 2, 2); show_tensor_img(content_rec, 'Content VAE recon')
plt.subplot(2, 2, 3); show_pil_img(style_pil, 'Style original')
plt.subplot(2, 2, 4); show_tensor_img(style_rec, 'Style VAE recon')
plt.tight_layout()
plt.show()


## Scale Fusion Helper

A mixed token pyramid keeps content tokens at most scales and uses style tokens at selected scales.

Example: `style_scales=[7, 8, 9]` means keep content at coarse/middle scales and use style only at fine scales.

In [ ]:
def mix_idx_by_scale(content_idx_Bl, style_idx_Bl, style_scales):
    style_scales = set(style_scales)
    return [
        style_idx if scale_id in style_scales else content_idx
        for scale_id, (content_idx, style_idx) in enumerate(zip(content_idx_Bl, style_idx_Bl))
    ]

def decode_mixed_scales(style_scales):
    mixed_idx_Bl = mix_idx_by_scale(content_idx_Bl, style_idx_Bl, style_scales)
    with torch.no_grad():
        mixed_img = vae.idxBl_to_img(mixed_idx_Bl, same_shape=True, last_one=True)
    return mixed_img

def save_tensor_image(x, path):
    img = tensor_to_pil(x)
    path.parent.mkdir(parents=True, exist_ok=True)
    img.save(path)
    return path

def show_scale_results(experiments, title, save_name=None, ncols=4):
    images = []
    labels = []
    for label, scales in experiments:
        images.append(decode_mixed_scales(scales))
        labels.append(label)

    n = len(images)
    ncols = min(ncols, n)
    nrows = (n + ncols - 1) // ncols
    plt.figure(figsize=(4 * ncols, 4 * nrows))
    for i, (img, label) in enumerate(zip(images, labels)):
        plt.subplot(nrows, ncols, i + 1)
        show_tensor_img(img, label)
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

    if save_name is not None:
        grid = make_grid(torch.cat(images, dim=0).clamp(-1, 1).add(1).div(2), nrow=ncols, padding=2)
        grid = grid.detach().cpu().permute(1, 2, 0).numpy()
        out_path = OUT_DIR / save_name
        Image.fromarray((grid * 255).astype('uint8')).save(out_path)
        print('saved:', out_path)

    return images


## Ablation 1: Coarse, Middle, Fine Scale Replacement

This is the main sanity check.

- Coarse replacement asks whether global layout changes.
- Middle replacement asks whether color/material/shape changes.
- Fine replacement asks whether texture changes while content stays recognizable.

In [ ]:
scale_group_experiments = [
    ('content only', []),
    ('style coarse 0-2', [0, 1, 2]),
    ('style middle 3-6', [3, 4, 5, 6]),
    ('style fine 7-9', [7, 8, 9]),
    ('style all 0-9', list(range(len(patch_nums)))),
]

_ = show_scale_results(
    scale_group_experiments,
    title='Content tokens mixed with style token scales',
    save_name='scale_groups.png',
    ncols=5,
)


## Ablation 2: Progressive Fine-To-Middle Style Injection

This tests how early we can start replacing content scales before content identity collapses.

In [ ]:
progressive_experiments = [
    ('content only', []),
    ('style scale 9', [9]),
    ('style scales 8-9', [8, 9]),
    ('style scales 7-9', [7, 8, 9]),
    ('style scales 6-9', [6, 7, 8, 9]),
    ('style scales 5-9', [5, 6, 7, 8, 9]),
    ('style scales 4-9', [4, 5, 6, 7, 8, 9]),
    ('style scales 3-9', [3, 4, 5, 6, 7, 8, 9]),
]

_ = show_scale_results(
    progressive_experiments,
    title='Progressively replace more scales with style tokens',
    save_name='progressive_fine_to_middle.png',
    ncols=4,
)


## Ablation 3: Single-Scale Replacement

This tests one scale at a time. It helps identify the first scale where style visibly changes the output and the first scale where content starts to break.

In [ ]:
single_scale_experiments = [('content only', [])]
for scale_id, pn in enumerate(patch_nums):
    single_scale_experiments.append((f'style scale {scale_id} ({pn}x{pn})', [scale_id]))

_ = show_scale_results(
    single_scale_experiments,
    title='Single-scale style token replacement',
    save_name='single_scale_sweep.png',
    ncols=4,
)


## Try More Style Types

Run the cell below to compare the same content image against several styles using the fine-scale replacement rule.

In [ ]:
multi_style_examples = [
    ('VanGogh', STYLE_DIR / 'VanGogh' / 'VanGogh001.png'),
    ('Monet', STYLE_DIR / 'Monet' / 'Monet001.png'),
    ('WaterColor', STYLE_DIR / 'WaterColor' / 'WC001.png'),
    ('Sketch', STYLE_DIR / 'Sketch' / 'S001.png'),
    ('PixelArt', STYLE_DIR / 'PixelArt' / 'PA001.png'),
]

style_scales = [7, 8, 9]
plt.figure(figsize=(15, 9))

for col, (style_name, path) in enumerate(multi_style_examples):
    style_x_i, style_pil_i = load_var_image(path)
    with torch.no_grad():
        style_idx_i = vae.img_to_idxBl(style_x_i)
        mixed_idx_i = mix_idx_by_scale(content_idx_Bl, style_idx_i, style_scales)
        mixed_img_i = vae.idxBl_to_img(mixed_idx_i, same_shape=True, last_one=True)

    plt.subplot(3, len(multi_style_examples), col + 1)
    show_pil_img(style_pil_i, f'{style_name} style')

    plt.subplot(3, len(multi_style_examples), col + 1 + len(multi_style_examples))
    show_tensor_img(mixed_img_i, f'{style_name} mix 7-9')

    plt.subplot(3, len(multi_style_examples), col + 1 + 2 * len(multi_style_examples))
    show_tensor_img(content_rec, 'content recon')

plt.tight_layout()
plt.show()


## What To Record

For each output, write down:

- Does the object/content identity stay recognizable?
- Does the output inherit color palette from the style image?
- Does it inherit brush/line/texture from the style image?
- Does the style image structure leak into the content image?

This notebook should give us the first experimental evidence for which scales are safe candidates for later VAR transformer style injection.